# T1 — Two rhythms: rotation number and locking

**Facts used** (classical; module T1; compendium C8/C9/C10 reimplemented
here in Python).

1. For the sine circle map $\theta \mapsto \theta + \Omega - (K/2\pi)\sin 2\pi\theta$
   the rotation number $\rho(\Omega) = \lim (\theta_n - \theta_0)/n$ exists, is
   monotone non-decreasing in $\Omega$, and locks on intervals at every
   rational (Poincaré; Arnold 1961; Jensen–Bak–Bohr 1983).
2. Tongue widths scale as $w(p/q) \propto K^q$ for small $K$ (Arnold 1961);
   hence $d\ln(w_{1/2}/w_{1/3})/d\ln K \to -1$ (compendium C9).
3. Adler (1946): $\varphi' = \Delta\omega - 2K\sin\varphi$ locks iff
   $|\Delta\omega| \le 2K$; below threshold the beat frequency is
   $\sqrt{\Delta\omega^2 - (2K)^2}$ (compendium C10; catalog c23 states the
   same fact with coupling $K$ in place of $2K$).

The rationals are the lockable ratios; the tongue edges are found below by
the tangency condition rather than by scanning, so the widths are derived
from the map and not read off a grid.

In [ ]:
import sys, math, json, cmath, random
from fractions import Fraction
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CATALOG.md").exists())
sys.path.insert(0, str(_root / "notebooks"))
from nbkit import ROOT, show_svg, catalog, verify, mutant_must_fail, falsify
import termplot
print("repo root:", ROOT.name)

In [ ]:
def step(th, Om, K):
    return th + Om - (K / (2 * math.pi)) * math.sin(2 * math.pi * th)

def rho(Om, K, iters=900, transient=250):
    th = 0.0
    for _ in range(transient):
        th = step(th, Om, K)
    start = th
    for _ in range(iters):
        th = step(th, Om, K)
    return (th - start) / iters

K = 1.0
grid = [(i / 200, rho(i / 200, K)) for i in range(201)]
mono = all(b[1] >= a[1] - 1e-3 for a, b in zip(grid, grid[1:]))
print("monotone on a 200-point grid:", mono)
print(termplot.staircase(grid, width=64, height=18, title="rho(Omega) at K = 1: the devil's staircase",
                         xlabel="Omega", ylabel="rho", markers={"1/2": 0.5, "1/3": 1 / 3, "2/3": 2 / 3}))

## Tongue edges by tangency

$\rho = p/q$ is locked iff $g(\theta) = f^q(\theta) - \theta - p$ has a zero.
$g$ increases with $\Omega$ for every $\theta$, so the left edge is the
$\Omega$ where $\max_\theta g = 0$ and the right edge where $\min_\theta g = 0$.

In [ ]:
def g_extrema(Om, K, p, q, n=720):
    lo, hi = float("inf"), float("-inf")
    for i in range(n):
        th0 = i / n
        th = th0
        for _ in range(q):
            th = step(th, Om, K)
        v = th - th0 - p
        lo, hi = min(lo, v), max(hi, v)
    return lo, hi

def tongue(p, q, K, span=0.02, scan=200):
    target = p / q
    # the tongue need not contain p/q itself (its centre shifts by O(K^2) for q >= 3):
    # scan for one locked Omega, then bisect each edge from there
    inside = None
    for i in range(scan + 1):
        Om = target - span + 2 * span * i / scan
        lo, hi = g_extrema(Om, K, p, q, n=360)
        if lo <= 0 <= hi:
            inside = Om; break
    assert inside is not None, "no locked point found in the scan"
    def bisect(fn, a, b):               # fn increasing in Om, fn(a) < 0 <= fn(b)
        for _ in range(40):
            m = (a + b) / 2
            if fn(m) < 0: a = m
            else: b = m
        return (a + b) / 2
    left = bisect(lambda Om: g_extrema(Om, K, p, q)[1], inside - span, inside)
    right = bisect(lambda Om: g_extrema(Om, K, p, q)[0], inside, inside + span)
    return left, right

widths = {}
for K in (0.2, 0.4):
    for p, q in ((1, 2), (1, 3)):
        l, r = tongue(p, q, K)
        widths[(K, q)] = r - l
        print(f"K = {K}: tongue {p}/{q} = [{l:.6f}, {r:.6f}]  width {r - l:.4e}")
print("compendium C9 reference widths: 3.147e-3, 1.2533e-2 (1/2); 2.83e-4, 2.183e-3 (1/3)")

In [ ]:
def check(q_offset=0, slope_target=-1.0):
    r12 = widths[(0.4, 2)] / widths[(0.2, 2)]
    r13 = widths[(0.4, 3)] / widths[(0.2, 3)]
    abs_ok = abs(r12 - 2 ** (2 + q_offset)) / 2 ** (2 + q_offset) < 0.15 and \
             abs(r13 - 2 ** (3 + q_offset)) / 2 ** (3 + q_offset) < 0.15
    slope = math.log((widths[(0.2, 2)] / widths[(0.2, 3)]) / (widths[(0.4, 2)] / widths[(0.4, 3)])) / math.log(0.2 / 0.4)
    print(f"  doubling K: w(1/2) x{r12:.2f}, w(1/3) x{r13:.2f}; ratio-law slope {slope:.3f}")
    return abs_ok and abs(slope - slope_target) < 0.15

falsify(check, {"exponent-q-plus-1": lambda: {"q_offset": 1},
                "ratio-law-slope-zero": lambda: {"slope_target": 0.0}})

## Adler's threshold

In [ ]:
def beat(dw, K, coef=2.0, T=400.0, dt=0.002):
    # integrate phi' = dw - coef K sin phi with RK4; count rotations. Lock must emerge, not be assumed.
    f = lambda p: dw - coef * K * math.sin(p)
    phi, rot, last = 0.0, 0, 0.0
    n = int(T / dt)
    for _ in range(n):
        k1 = f(phi); k2 = f(phi + dt * k1 / 2); k3 = f(phi + dt * k2 / 2); k4 = f(phi + dt * k3)
        phi += dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6
        while phi - last > 2 * math.pi:
            rot += 1; last += 2 * math.pi
    return rot * 2 * math.pi / T

K = 0.3
rows = []
for dw in (0.45, 0.6, 0.62, 0.7, 0.9):
    b = beat(dw, K)
    pred = 0.0 if abs(dw) <= 2 * K else math.sqrt(dw * dw - 4 * K * K)
    rows.append((dw, b, pred))
    print(f"K = {K}, dw = {dw:.2f}: beat {b:.4f}   sqrt(dw^2 - (2K)^2) = {pred:.4f}")
print(termplot.plot_xy([(dw, b) for dw, b, _ in rows] + [(dw, p) for dw, _, p in rows], width=50, height=10,
                       title="beat frequency vs dw (measured and formula)", xlabel="dw", ylabel="beat"))

In [ ]:
def check(thr_factor=2.0, coef=2.0):
    thr = thr_factor * K
    pred = lambda dw: 0.0 if abs(dw) <= thr else math.sqrt(dw * dw - thr * thr)
    b1, b2, b3 = beat(0.45, K, coef), beat(0.62, K, coef), beat(0.9, K, coef)
    return b1 == pred(0.45) and b2 > 0 and abs(b2 - pred(0.62)) < 0.02 and abs(b3 - pred(0.9)) < 0.02

falsify(check, {"threshold-K-not-2K": lambda: {"thr_factor": 1.0},
                "ode-coefficient-K": lambda: {"coef": 1.0}})

The golden ratio is the most stubborn drifter: the gaps in the staircase
at $K<1$ are the irrationals, and the one worst approximated by rationals
(Hurwitz) is $(\sqrt5 - 1)/2$. Not re-verified here; stated with citation only.

## Falsifier: catalog c23 (Adler, in the $\delta - K\sin\theta$ convention) must fail under its mutant

In [ ]:
rc, _ = catalog("c23_adler_locking_range")
assert rc == 0
rc, out = catalog("c23_adler_locking_range", mutant=True)
mutant_must_fail("c23_adler_locking_range", rc, out)